# 2. A real room: from a 3D scan to TouchDesigner

The scan of Palazzo Datini (`data/datini_25k.stl`). `analyze` does the whole chain:

1. reads the surface, drops scan debris, orients the normals;
2. computes the Schroeder frequency from volume, surfaces and materials, and uses it as `f_max`;
3. builds the volume mesh. gmsh needs a clean surface; this scan intersects itself in a few
   places, so the mesh comes from fTetWild, which accepts it;
4. computes all modes up to `f_max` and the decay of each one.

In [ ]:
import numpy as np
import plotly.graph_objects as go

import tdroommodes as trm
from tdroommodes import synth

In [ ]:
absorption = {"floor": "wood_floor", "walls": "plaster_lime_rough", "ceiling": "wood_panel_thin"}
result = trm.analyze("../data/datini_25k.stl", absorption=absorption, up="z")

s = result.statistics
print(f"volume {s['volume_m3']:.0f} m³, surface {s['surface_area_m2']:.0f} m²")
print(f"Schroeder frequency {s['schroeder_frequency_hz']:.1f} Hz")
print(f"{len(result.modes)} modes, {result.modes.info['dofs']} unknowns")
print({k: f"{v:.1f} s" for k, v in result.seconds.items()})

`up` is the vertical axis of the scan: it decides which surfaces are floor and ceiling.
Check the areas: a floor much smaller than the walls usually means the wrong axis
(`datini_15k.stl`, for example, is rotated).

In [ ]:
{k: round(v) for k, v in s["surfaces_m2"].items()}

In [ ]:
trm.plots.spectrum(result.modes, result.source, result.receiver, s["schroeder_frequency_hz"])

The lowest mode, near 3.6 Hz, is far below the first mode along the length of the hall
(about 7 Hz). The pressure is almost uniform in the hall and swings against a small cavity:
a Helmholtz resonance, like blowing over a bottle. The FEM is right for this geometry; whether
that cavity exists in the building or is a gap in the scan is worth checking.

In [ ]:
first = result.modes.evaluate(result.room.mesh.p.T)[:, 0]
print("cavity near", result.room.mesh.p[:, np.argsort(first**2)[-50:]].mean(axis=1).round(1), "m")

In [ ]:
trm.plots.gallery(result.modes, axis="z", max_modes=8)

## Animation

A source playing a sine excites every mode, most of all those near its frequency. The field is
a weighted sum of the precomputed shapes, so each frame is one matrix product: this is what
makes real-time use possible.

In [ ]:
u, v, slice_shapes = trm.plots.slice_values(result.modes, axis="z", resolution=0.25)
shapes = np.moveaxis(slice_shapes, 0, -1)          # (rows, columns, modes)
m = result.modes
phi_source = m.evaluate([result.source])[0]

frequency = m.frequencies[5]
field = synth.steady_state(frequency, shapes, m.frequencies, m.decay, phi_source, m.c)
scale = np.nanmax(np.abs(field))
times = np.linspace(0, 1 / frequency, 24, endpoint=False)
frames = [go.Frame(data=[go.Heatmap(z=synth.steady_state_frame(field, frequency, t) / scale)]) for t in times]

fig = go.Figure(
    data=[go.Heatmap(x=u, y=v, z=synth.steady_state_frame(field, frequency, 0) / scale,
                     zmin=-1, zmax=1, colorscale="RdBu_r")],
    frames=frames,
    layout=go.Layout(title=f"Source at {frequency:.1f} Hz, one period", height=500,
                     yaxis=dict(scaleanchor="x"),
                     updatemenus=[dict(type="buttons", buttons=[dict(label="Play", method="animate",
                         args=[None, dict(frame=dict(duration=60), fromcurrent=True, mode="immediate")])])]))
fig

## Export for TouchDesigner

`grid.npz` holds the shapes on a regular 3D grid (`shapes`: x × y × z × modes, zero outside the
room), with `frequencies`, `decay` and the axes. It needs only numpy to read, and TouchDesigner
ships numpy. `modes.npz` holds the same on the vertices of the mesh (`vertices`,
`vertex_shapes`), for point clouds.

In [ ]:
out = result.write("../results/datini", grid_spacing=0.5)
grid = synth.load(out / "grid.npz")
print(sorted(grid), grid["shapes"].shape)

## In TouchDesigner

The folder `touchdesigner/` holds the callbacks of three Script operators and a setup guide
(`touchdesigner/README.md`): a Script CHOP with the weight of each mode, a Script TOP with a slice
of the field, a Script TOP with a point cloud for instancing. They all run
`tdroommodes.runtime`, which imports only numpy. Here is the same computation outside
TouchDesigner: an impulse at the source, 50 ms later, on the horizontal plane at mid height.

In [ ]:
from tdroommodes import runtime

room = runtime.load(out / "grid.npz")
weights = room.weights("impulse", t=0.05, source=result.source)
field = room.slice(weights, axis="z", position=0.5)
go.Figure(go.Heatmap(x=room.x, y=room.y, z=field, zmin=-1, zmax=1, colorscale="RdBu_r")).update_layout(
    height=450, yaxis=dict(scaleanchor="x"), title="Impulse at the source, t = 50 ms")